# AI Leadership Insight Agent

This notebook implements an AI-powered assistant for leadership that answers questions about company status and performance using internal documents.

## Overview
- Ingests and processes company documents (annual reports, quarterly reports, strategy notes, etc.)
- Retrieves relevant information for natural language questions
- Generates concise, factual answers grounded in the documents
- Includes visualizations and validation

## Assumptions
- Using Python for implementation
- OpenAI models for embeddings (text-embedding-3-small) and LLM (gpt-4o-mini)
- FAISS for vector storage
- Documents are in text format (can be extended to PDF)
- Answers are grounded in retrieved context
- No specific metrics required; validation based on dataset and queries

In [ ]:
# Import Required Libraries
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt

from config import *
from src.document_loader import load_documents
from src.chunking import chunk_text
from src.embeddings import get_embeddings
from src.vector_store import build_index
from src.retrieval import retrieve
from src.llm import generate_answer

In [ ]:
# Load and Process Documents
print("Loading documents...")
docs = load_documents(DATA_PATH)
print(f"Loaded {len(docs)} documents")
# Display sample document content
if docs:
    print("Sample document content:")
    print(docs[0][:500] + "...")

In [ ]:
# Chunk Text
print("Chunking documents...")
all_chunks = []
for doc in docs:
    all_chunks.extend(chunk_text(doc, CHUNK_SIZE, CHUNK_OVERLAP))
print(f"Created {len(all_chunks)} chunks")
# Display sample chunk
if all_chunks:
    print("Sample chunk:")
    print(all_chunks[0][:200] + "...")

In [ ]:
# Generate Embeddings
print("Creating embeddings...")
chunk_embeddings = get_embeddings(all_chunks, EMBEDDING_MODEL)
print(f"Generated embeddings for {len(chunk_embeddings)} chunks")
# Note: Requires OpenAI API key in config

In [ ]:
# Build Vector Index
print("Building vector index...")
index = build_index(chunk_embeddings)
print("Vector index built using FAISS")

In [ ]:
# Retrieve Relevant Context
def retrieve_context(question):
    print("Embedding question...")
    question_embedding = get_embeddings([question], EMBEDDING_MODEL)[0]
    print("Retrieving relevant context...")
    relevant_chunks = retrieve(question_embedding, index, all_chunks, TOP_K)
    context = "\n\n".join(relevant_chunks)
    return context

In [ ]:
# Generate Answers with LLM
def generate_answer_func(question, context):
    print("Generating answer...")
    answer = generate_answer(LLM_MODEL, context, question)
    return answer

In [ ]:
# Test with Sample Questions
questions = [
    "What is our current revenue trend?",
    "Which risks were highlighted?",
    "What are the key strategic priorities?"
]

for question in questions:
    print(f"\nQUESTION: {question}")
    context = retrieve_context(question)
    answer = generate_answer_func(question, context)
    print("ANSWER:")
    print(answer)
    print("-" * 80)

In [ ]:
# Visualize Performance Metrics
import matplotlib.pyplot as plt

chunk_lengths = [len(chunk) for chunk in all_chunks]
plt.figure(figsize=(8, 5))
plt.hist(chunk_lengths, bins=20, alpha=0.7)
plt.title('Distribution of Chunk Lengths')
plt.xlabel('Chunk Length (characters)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Note: For more advanced metrics, we could evaluate retrieval accuracy or answer quality with a validation set

# Validation and Assumptions

## Validation
- Tested on sample questions: "What is our current revenue trend?", "Which risks were highlighted?", "What are the key strategic priorities?"
- Outputs are natural language responses with appropriate details and report sections
- Validation based on dataset and queries; answers grounded in retrieved document chunks
- No specific metrics required by Adobe; manual evaluation of factual accuracy

## Assumptions
- Using Python as required
- Any LLM/embedding approach: OpenAI GPT-4o-mini for generation, text-embedding-3-small for embeddings
- Documents ingested from folder (text/PDF supported)
- Retrieval uses FAISS vector index for efficiency
- Answers are concise and factual, based on context
- Agent handles open-ended questions by retrieving relevant info
- Can be extended to autonomous decision-making in future

## How to Run and Test
1. Ensure OpenAI API key is set in `config.py` (e.g., `OPENAI_API_KEY = "your_key_here"`)
2. Install dependencies: `pip install -r requirements.txt`
3. Place documents in `data/sample_docs/`
4. Run the notebook cells sequentially
5. For script version, run `python main.py`
6. Test with custom questions by modifying the `questions` list

This notebook serves as a complete submission demonstrating the AI Leadership Insight Agent.